In [ ]:
# models tested = 
    #"mistral-nemo:latest", ok, GOOD
    #"llama3.1:latest", ok but do not follow the instruction
    #"internlm2:latest", # no good, it is hanging up and not responding
    #"gemma2:latest", # none sens output
    #"qwen2:latest", GOOD
    #"phi3:latest", failing on part 3
    #"mistral:latest", GOOD
    #"llama3:latest", no good, it is hanging up and not responding
    #"llama3-gradient:latest",no good, it is hanging up and not responding
    #"mixtral:latest",,no good. too much ram


In [5]:
import ollama
from nltk import word_tokenize
import re
import math
import os
from magic_doc.docconv import DocConverter

converter = DocConverter(s3_config=None)
markdown_content, time_cost = converter.convert("/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/Binder1.pdf", conv_timeout=300)

# list of the latest LLM models tested and good for the task: # 1 'qwen2', # 2 'mistral', # 3 'mistral-nemo'
# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'qwen2', # 'qwen2', 'mistral', 'mistral-nemo'
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'qwen2',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'qwen2',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'mistral',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    print("Splitting the document into sections...")
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")

    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")

    print("Document split into sections successfully.")
    return flattened_sections

def calculate_context_window(sections):
    print("Calculating context window...")
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    print(f"Context window calculated: {context_window}")
    print("Starting evaluation process...")
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    print(f"Analyzing section '{section_title}' with personality '{personality_key}'...")
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    print(f"Analysis for section '{section_title}' with personality '{personality_key}' completed.")
    return response_content

def evaluate_all_experts(document_text):
    print("Evaluating all experts...")
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in zip(section_titles, sections):
            print(f"The {personality_key} is analyzing the {section_title} section")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)

    print("All experts evaluated successfully.")
    return all_expert_responses

def analyze_reviewer(section_title, all_expert_responses):
    print(f"Analyzing reviewer for section '{section_title}'...")
    context_window = 5000

    # Extract the consolidated feedback for the given section
    consolidated_feedback = ""
    for expert in all_expert_responses:
        if section_title in expert["responses"]:
            consolidated_feedback += f"Personality: {expert['personality']}\nResponse:\n{expert['responses'][section_title]}\n\n"

    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="qwen2", messages=messages, options=options)
    response_content = response['message']['content']
    print(f"Reviewer analysis for section '{section_title}' completed.")
    return response_content

def combine_reviews(final_review, all_expert_responses):
    print("Combining reviews...")
    # Combine the reviews into a single string
    combined_content = []

    combined_content.append("# Final Review and Combined Review\n\n")

    # Append final review content
    combined_content.append("## Final Review\n")
    combined_content.append(final_review)
    combined_content.append("\n\n")

    # Append combined review content
    combined_content.append("## Combined Review\n")

    # Convert the dictionary to a string
    for expert in all_expert_responses:
        combined_content.append(f"### {expert['personality']}\n")
        for section, response in expert["responses"].items():
            combined_content.append(f"#### {section}\n")
            combined_content.append(response)
            combined_content.append("\n\n")

    combined_content.append("\n")

    print("Reviews combined successfully.")
    return "\n".join(combined_content)

def save_to_markdown(content, filepath):
    print(f"Saving combined content to {filepath}...")
    # Save the combined content to a markdown file
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(content)
    print(f"Combined review saved to {os.path.abspath(filepath)}")

document_text = markdown_content  # Assign your document content here
print("Starting text loading process...")
all_expert_responses = evaluate_all_experts(document_text)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, all_expert_responses))


# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = all_expert_responses  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, all_expert_responses)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print("Evaluation process completed.")


2024-08-05 15:33:15.080 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:57 - cid_count: 0, text_len: 45225, cid_chars_radio: 0.0
2024-08-05 15:33:15.163 | INFO     | magic_doc.contrib.pdf.pdf_extractor:run:70 - stream io data is digital pdf


Starting text loading process...
Evaluating all experts...
Splitting the document into sections...
Document split into sections successfully.
Calculating context window...
Context window calculated: 20000
Starting evaluation process...
The Highly analytical evaluator is analyzing the Introduction and Excellence section
Analyzing section 'Introduction and Excellence' with personality 'Highly analytical evaluator'...
Analysis for section 'Introduction and Excellence' with personality 'Highly analytical evaluator' completed.
The Highly analytical evaluator is analyzing the Impact section
Analyzing section 'Impact' with personality 'Highly analytical evaluator'...
Analysis for section 'Impact' with personality 'Highly analytical evaluator' completed.
The Highly analytical evaluator is analyzing the Quality and Efficiency of the Implementation section
Analyzing section 'Quality and Efficiency of the Implementation' with personality 'Highly analytical evaluator'...
Analysis for section 'Qual